# BehaviorTracker · OFT Rearing Training V1

Treinamento supervisionado com câmera superior. Nenhuma célula foi executada na criação deste notebook. Use anotações humanas e vídeos de animais independentes. O código de treinamento deve vir de uma cópia versionada e confiável do repositório.

In [ ]:
# 01 · Setup: Python 3.10–3.12 no Colab; instalação exclusiva de treinamento.
from google.colab import drive
drive.mount('/content/drive')
%pip install torch==2.5.1 torchvision==0.20.1 onnxruntime==1.20.1 onnx==1.20.1 scikit-learn==1.6.1 opencv-python-headless==4.13.0.92
from pathlib import Path
import subprocess, random, numpy as np, torch
REPO = Path('/content/behaviortracker') # Disponibilize aqui a versão de código revisada.
DATA = Path('/content/drive/MyDrive/BehaviorTracker/OFT_Rearing')
SNAPSHOT = DATA/'dataset/snapshot_v1'
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert (REPO/'scripts/behavior/build_oft_dataset.py').is_file(), 'Disponibilize a cópia versionada do repositório.'
def run(script,*args):
    subprocess.run(['python',str(REPO/'scripts/behavior'/script),*map(str,args)],check=True)


In [ ]:
# 02 · Verificação de vídeos e anotações humanas.
import sys, csv
sys.path.insert(0,str(REPO/'scripts/behavior'))
from build_oft_dataset import read_intervals, split_videos
from behavior.open_field import sha256
videos=list(csv.DictReader((DATA/'annotations/videos.csv').open()))
intervals=read_intervals(DATA/'annotations/intervals.csv')
for v in videos:
    assert sha256(DATA/'videos'/v.get('filename',v['video_id']+'.mp4')) == v['sha256']
print('Vídeos:',len(videos),'Intervalos humanos:',sum(map(len,intervals.values())))


In [ ]:
# 03 · Importação dos resultados de tracking existentes.
import json
from behavior.open_field import read_samples
for v in videos:
    tracking=DATA/'tracking'/v['video_id']
    request=json.loads((tracking/'request.json').read_text())
    assert request['config']['protocol']=='open_field'
    count=sum(1 for _ in read_samples(tracking/'samples.jsonl'))
    print(v['video_id'],count,'frames PTS')


In [ ]:
# 04 · Extração sequencial de crops, geometria, movimento e máscaras.
run('build_oft_dataset.py',DATA,SNAPSHOT,'--seed',SEED,'--rate',15)


In [ ]:
# 05 · Auditoria das partições por animal, sessão e SHA-256.
from common import load_dataset
_,rows=load_dataset(SNAPSHOT)
print(json.loads((SNAPSHOT/'splits.json').read_text()))
for split in ['train','validation','test']:
    print(split, sum(r['split']==split and r['label']!='UNKNOWN' for r in rows),'amostras humanas')


In [ ]:
# 06 · Modelo A: Random Forest.
run('train_oft_baseline.py',SNAPSHOT,DATA/'checkpoints/model_a','--seed',SEED)


In [ ]:
# 07 · Modelo B: MobileNetV3-Small supervisionada.
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
run('train_oft_visual.py',SNAPSHOT,DATA/'checkpoints/model_b','--architecture','visual','--device',DEVICE,'--seed',SEED)


In [ ]:
# 08 · Modelos C e D: TCN visual e híbrida na mesma divisão.
for name,architecture in [('model_c','visual_tcn'),('model_d','hybrid_tcn')]:
    run('train_oft_visual.py',SNAPSHOT,DATA/'checkpoints'/name,'--architecture',architecture,'--device',DEVICE,'--seed',SEED)


In [ ]:
# 09 · Exportações temporárias para inferência reproduzível e validação.
# Selecione arquitetura/limiares usando validation. Não escolha com base em test.
for name in ['model_a','model_b','model_c','model_d']:
    run('export_onnx.py',DATA/'checkpoints'/name,DATA/'export'/name,'--version','1.0.0-experimental')
    run('validate_oft_behavior.py',SNAPSHOT,DATA/'export'/name,DATA/'reports'/f'{name}_validation','--split','validation')


In [ ]:
# 10 · Vídeos de falsos positivos e negativos (revisão qualitativa).
from validate_oft_behavior import render_error_clips
dataset_manifest=json.loads((SNAPSHOT/'dataset_manifest.json').read_text())
for name in ['model_a','model_b','model_c','model_d']:
    report_dir=DATA/'reports'/f'{name}_validation'
    report=json.loads((report_dir/'validation_report.json').read_text())
    errors=json.loads((report_dir/'error_intervals.json').read_text())
    clips=render_error_clips(errors,dataset_manifest,DATA/'videos',report_dir/'error_clips',limit=30)
    print(name,report['allSessionsMeetDevelopmentTargets'],len(errors),'intervalos de erro',len(clips),'clipes')
# Reveja os clipes; as previsões não são ground truth e não substituem anotações humanas.


In [ ]:
# 11 · Teste independente após congelar modelo e decoder.
SELECTED_MODEL='model_d' # Decida usando validation, não test.
run('validate_oft_behavior.py',SNAPSHOT,DATA/'export'/SELECTED_MODEL,DATA/'reports'/f'{SELECTED_MODEL}_test','--split','test')
# O manifesto inclui hashes/tamanhos e paridade numérica; não publica release.
# Registre relatório, condições cobertas e versão do decoder antes da distribuição.


In [ ]:
# 12 · Benchmark local CPU e checagens da exportação.
from behavior.model_registry import load_manifest
manifest=load_manifest(DATA/'export'/SELECTED_MODEL/'manifest.json')
print('Equivalência numérica:',manifest['numericalParity'])
print('Leve o bundle ao computador de referência. Use benchmark_oft_cpu.py com um request.json local.')
print('Meça separadamente tracking, decode, crops, features, encoder, TCN, decoder, memória e threads.')
print('INT8 exige comparação de precisão com FP32 nos mesmos vídeos independentes antes de distribuição.')
